# Model Evaluation Notebook

This notebook demonstrates how to run the model evaluation from the `eval_multiple.py` script within a Jupyter environment.

In [ ]:
from typing import Any, Dict, List, Tuple

import hydra
import rootutils
from lightning import LightningDataModule, LightningModule, Trainer
from omegaconf import DictConfig, OmegaConf
import glob
import os
import numpy as np
import torch
import yaml

# Set environment variables and precision
os.environ['HYDRA_FULL_ERROR'] = '1'
torch.set_float32_matmul_precision('highest')

# Setup root directory for the project
root_dir = rootutils.find_root(search_from=os.path.dirname(os.getcwd()), indicator=".project-root")
rootutils.setup_root(root_dir, indicator=".project-root", pythonpath=True)

from src.utils import (
    RankedLogger,
    extras,
    instantiate_loggers,
    log_hyperparameters,
    task_wrapper,
)
from src.utils.metrics import calculate_metrics,calculate_metrics_time

In [ ]:
def mean_logits_per_oid(oids, logits, labels):
    """Mean logits per object id.

    :param oids: Object IDs.
    :param logits: Logits.
    :param labels: Labels.
    :return: Mean logits per object id.
    """
    # Ensure all elements in oids are strings
    list_oids = []
    for oid in oids:
        if isinstance(oid, bytes):
            list_oids.append(oid.decode('utf-8').split('_')[0])
        elif isinstance(oid, str):
            list_oids.append(oid.split('_')[0])
        else:
            raise TypeError(f"Invalid type for oid: {type(oid)}")
    list_oids = np.array(list_oids)
    # Create a dictionary with oids, logits, labels
    dict_logits = {}
    for i in range(len(list_oids)):
        if list_oids[i] not in dict_logits:
            dict_logits[list_oids[i]] = {'logits': [], 'labels': []}
        dict_logits[list_oids[i]]['logits'].append(logits[i])
        dict_logits[list_oids[i]]['labels'].append(labels[i])
    # Mean logits and labels
    for key in dict_logits:
        dict_logits[key]['logits'] = np.argmax(np.mean(np.array(dict_logits[key]['logits']), axis=0))
        dict_logits[key]['labels'] = np.mean(np.array(dict_logits[key]['labels']), axis=0)
    return dict_logits



In [ ]:
from matplotlib import cm


log = RankedLogger(__name__, rank_zero_only=True)


def evaluate(cfg: DictConfig, test_type: str, max_time_to_eval: int = None) -> Tuple[Dict[str, Any], Dict[str, Any]]:
    """Evaluates given checkpoint on a datamodule testset.

    :param cfg: DictConfig configuration composed by Hydra.
    :param test_type: Type of test set to evaluate.
    :return: Tuple[dict, dict] with metrics and dict with all instantiated objects.
    """
    assert cfg.ckpt_path, "Checkpoint path must be provided."
    # Disable all loggers by setting the logger config to an empty list or None
    cfg.trainer.logger = False

    cfg.data.test_set_type = test_type
    cfg.data.normalize_tab = True
    cfg.data.max_time_to_eval = max_time_to_eval
    cfg.data.return_snids = True
    trainer: Trainer = hydra.utils.instantiate(cfg.trainer)

    log.info("Starting testing!")

    # Find all .ckpt files recursively under cfg.ckpt_path, expecting structure like 0/checkpoints/xxx.ckpt, 1/checkpoints/xxx.ckpt, etc.
    ckpt_files = glob.glob(os.path.join(cfg.ckpt_path, "*", "checkpoints", "*.ckpt"))
    # Remove any checkpoint files that end with 'last.ckpt'
    ckpt_files = [f for f in ckpt_files if not f.endswith('last.ckpt')]
    # Sort by the integer folder name (e.g., 0, 1, 2, ...)
    ckpt_files_clean = []
    for ckpt in ckpt_files:
        with open(os.path.join(os.path.dirname(os.path.dirname(ckpt)), ".hydra", "config.yaml"), 'r') as f:
            config_yaml = yaml.safe_load(f)
        data_percentage = config_yaml['data'].get('percentage', 1)
        if data_percentage >= 1:
            ckpt_files_clean.append(ckpt)
    
    print(len(ckpt_files_clean), "checkpoints found for evaluation.")
    #data splits
    ckpt_files = ckpt_files_clean
    data_splits = []
    for ckpt in ckpt_files:
        with open(os.path.join(os.path.dirname(os.path.dirname(ckpt)), ".hydra", "config.yaml"), 'r') as f:
            config_yaml = yaml.safe_load(f)
        data_splits.append(config_yaml['data'].get('split', 0))
    all_targets_lcs, all_preds_lcs = [], []
    all_targets_feat, all_preds_feat = [], []
    all_targets_mix, all_preds_mix = [], []

    for split, ckpt in enumerate(ckpt_files):
        print(f"Evaluating split {split} with checkpoint {ckpt}")
        cfg.data.split = split % 5
        datamodule: LightningDataModule = hydra.utils.instantiate(cfg.data)
        model: LightningModule = hydra.utils.instantiate(cfg.model)

        out = trainer.predict(model=model, dataloaders=datamodule, ckpt_path=ckpt)

        model_logits_lcs,model_logits_feat,model_logits_mix, model_targets, model_oids = [], [], [], [], []

        for batch in out:
            model_targets.append(batch['targets'])
            model_logits_lcs.append(batch['logits_lc'])
            model_logits_feat.append(batch['logits_feat'])
            model_logits_mix.append(batch['logits_mix'])
            model_oids += batch['oid']
        if model_logits_lcs and model_logits_lcs[0] is not None:
            model_logits_lcs = torch.cat(model_logits_lcs, axis=0).to(torch.float32).detach().cpu().numpy()
        else:
            model_logits_lcs = None
        if model_logits_feat and model_logits_feat[0] is not None:
            model_logits_feat = torch.cat(model_logits_feat, axis=0).to(torch.float32).detach().cpu().numpy()
        else:
            model_logits_feat = None
        if model_logits_mix and model_logits_mix[0] is not None:
            model_logits_mix = torch.cat(model_logits_mix, axis=0).to(torch.float32).detach().cpu().numpy()
        else:
            model_logits_mix = None
        model_targets = torch.cat(model_targets, axis=0).detach().cpu().numpy()
        if model_logits_lcs is not None:
            matched_predictions_lcs = mean_logits_per_oid(oids=model_oids, logits=model_logits_lcs, labels=model_targets)
            all_targets_lcs.append(np.array([matched_predictions_lcs[key]['labels'] for key in matched_predictions_lcs]))
            all_preds_lcs.append(np.array([matched_predictions_lcs[key]['logits'] for key in matched_predictions_lcs]))
        else:
            all_preds_lcs = None
        if model_logits_feat is not None:
            matched_predictions_feat = mean_logits_per_oid(oids=model_oids, logits=model_logits_feat, labels=model_targets)
            all_targets_feat.append(np.array([matched_predictions_feat[key]['labels'] for key in matched_predictions_feat]))
            all_preds_feat.append(np.array([matched_predictions_feat[key]['logits'] for key in matched_predictions_feat]))
        else:
            all_preds_feat = None
    
        if model_logits_mix is not None:
            matched_predictions_mix = mean_logits_per_oid(oids=model_oids, logits=model_logits_mix, labels=model_targets)
            all_targets_mix.append(np.array([matched_predictions_mix[key]['labels'] for key in matched_predictions_mix]))
            all_preds_mix.append(np.array([matched_predictions_mix[key]['logits'] for key in matched_predictions_mix]))
        else:
            all_preds_mix = None
    

    if all_preds_lcs is not None:

        cm_title_lcs = cfg.get('cm_title')
        if cm_title_lcs:
            cm_title_lcs = cm_title_lcs.replace('MD-', '').replace('FT-', '')
        calculate_metrics_time(
            targets_list=all_targets_lcs,
            preds_list=all_preds_lcs,
            path=cfg.paths.output_dir,
            classes=cfg.get('classes'),
            classes_order=cfg.get('classes_order'),
            cm_title=cm_title_lcs,
            experiment_name= cfg.get('experiment_name', 'new_atat_windows'),
            all_experiments_csv= cfg.get('all_experiments_csv', None),
            modality= 'LC',
            max_time_to_eval=max_time_to_eval
        )
    if all_preds_feat is not None:
        cm_title_feat = cfg.get('cm_title')
        if cm_title_feat:
            cm_title_feat = cm_title_feat.replace('LC-', '')
        if 'FT' not in cm_title_feat:
            cm_title_feat = cm_title_feat.replace('-MTA', '')
        calculate_metrics_time(
            targets_list=all_targets_feat,
            preds_list=all_preds_feat,
            path=cfg.paths.output_dir,
            classes=cfg.get('classes'),
            classes_order=cfg.get('classes_order'),
            cm_title=cm_title_feat,
            experiment_name= cfg.get('experiment_name', 'new_atat_windows'),
            all_experiments_csv= cfg.get('all_experiments_csv', None),
            modality= 'Feat',
            max_time_to_eval=max_time_to_eval
        )
    if all_preds_mix is not None:
        calculate_metrics_time(
            targets_list=all_targets_mix,
            preds_list=all_preds_mix,
            path=cfg.paths.output_dir,
            classes=cfg.get('classes'),
            classes_order=cfg.get('classes_order'),
            cm_title=cfg.get('cm_title'),
            experiment_name= cfg.get('experiment_name', 'new_atat_windows'),
            all_experiments_csv= cfg.get('all_experiments_csv', None),
            modality= 'Mix',
            max_time_to_eval=max_time_to_eval
        )


In [ ]:
def run_several_exps(dict_exp):
    name = dict_exp['name']
    cm_title = dict_exp['cm_title']
    experiment_name = dict_exp['experiment_name']
    max_time_to_eval = dict_exp['max_time_to_eval']
    type = dict_exp['type']
    experiment_path = "../logs/" + type + '/' + name
    experiment_cfg = os.path.join(experiment_path, "multirun.yaml")
    all_experiments_path = '../logs/eval/ATATPeriodicComparisonTTE_' + type + '.csv'
    with open(experiment_cfg, "r") as f:
        cfg = yaml.safe_load(f)

    cfg = OmegaConf.create(cfg)
    # Remove the date/time suffix from the experiment path
    experiment_name_path = "_".join(experiment_path.split('/')[-1].split('_')[:-2])
    os.makedirs(os.path.join('/home/fsoto/Documents/LCsSSL/logs/eval', experiment_name_path), exist_ok=True)
    cfg.paths.output_dir = os.path.join('/home/fsoto/Documents/LCsSSL/logs/eval', experiment_name_path)
    cfg.ckpt_path = experiment_path

    # Dynamically load classes and baseline metrics
    #cfg.classes = [
    #    'AGN', 'EB/EW', 'Blazar', 'CEP', 'LPV', 'CV/Nova', 'Periodic-Other', 'Microlensing', 'QSO',
    #    'DSCT', 'EA', 'RRLab', 'RSCVn', 'RRLc', 'SLSN', 'SNII', 'SNIbc', 'SNIIn', 'SNIa', 'TDE', 'YSO'
    #]
    #cfg.classes_order = [
    #    'SNIa', 'SNIbc', 'SNII', 'SNIIn', 'SLSN', 'TDE', 'Microlensing', 'QSO', 'AGN', 'Blazar', 'YSO',
    #    'CV/Nova', 'LPV', 'EA', 'EB/EW', 'Periodic-Other', 'RSCVn', 'CEP', 'RRLab', 'RRLc', 'DSCT'
    #]
    cfg.classes = ['CEP', 'DSCT', 'EA', 'EB/EW', 'LPV', 'Periodic-Other', 'RRLab', 'RRLc', 'RSCVn']

    cfg.cm_title = cm_title
    cfg.trainer.devices = [1]  # Use only one GPU for evaluation
    cfg.experiment_name = experiment_name
    cfg.all_experiments_csv = all_experiments_path
    cfg = OmegaConf.merge(cfg, OmegaConf.from_dotlist([]))
    for i in max_time_to_eval:
        evaluate(cfg, test_type='test',max_time_to_eval=i)

In [ ]:
experiments = [
    {   'type': 'lc',
        'name': 'ATAT_Periodic_Lightcurves_2025-10-02_16-57-36',
        'cm_title': 'ATAT LC-MTA',
        'experiment_name': 'ATAT LC',
        'max_time_to_eval': [8, 16, 32, 64, 128, 256, 512, 1024, 2048]
    },
    {   'type': 'lc',
        'name': 'DiT_Periodic_Lightcurves_2025-10-21_14-40-20',
        'cm_title': 'DiT LC-MTA',
        'experiment_name': 'DiT LC',
        'max_time_to_eval': [8, 16, 32, 64, 128, 256, 512, 1024, 2048]
    },

    {   'type': 'lc',
        'name': 'DiT_Periodic_VICReg_Lightcurves_2025-10-21_20-18-44',
        'cm_title': 'DiT VICReg LC-MTA',
        'experiment_name': 'DiT_VICReg LC',
        'max_time_to_eval': [8, 16, 32, 64, 128, 256, 512, 1024, 2048]
    },
    {   'type': 'lc',
        'name': 'DiT_Periodic_VICReg_LinearClassification_2025-10-22_12-28-42',
        'cm_title': 'DiT VICReg LC-MTA-LP',
        'experiment_name': 'DiT_VICReg_Linear LC',
        'max_time_to_eval': [8, 16, 32, 64, 128, 256, 512, 1024, 2048]
    },
    {   'type': 'lc',
        'name': 'DiViT_L_Periodic_Lightcurves_2025-10-24_15-33-41',
        'cm_title': 'DiViT-L LC-MTA',
        'experiment_name': 'DiViT_L LC',
        'max_time_to_eval': [8, 16, 32, 64, 128, 256, 512, 1024, 2048]
    },
    {   'type': 'lc',
        'name': 'DiViT_L_Periodic_VICReg_Lightcurves_2025-10-24_20-42-35',
        'cm_title': 'DiViT-L VICReg LC-MTA',
        'experiment_name': 'DiViT_L_VICReg LC',
        'max_time_to_eval': [8, 16, 32, 64, 128, 256, 512, 1024, 2048]
    },
    {   'type': 'lc',
        'name': 'DiViT_L_Periodic_VICReg_LinearClassification_2025-10-25_11-08-43',
        'cm_title': 'DiViT-L VICReg LC-MTA-LP',
        'experiment_name': 'DiViT_L_VICReg_Linear LC',
        'max_time_to_eval': [8, 16, 32, 64, 128, 256, 512, 1024, 2048]
    },
    {   'type': 'lc',
        'name': 'DiViT_Periodic_Lightcurves_2025-10-23_15-25-10',
        'cm_title': 'DiViT LC-MTA',
        'experiment_name': 'DiViT LC',
        'max_time_to_eval': [8, 16, 32, 64, 128, 256, 512, 1024, 2048]
    },
    {   'type': 'lc',
        'name': 'DiViT_Periodic_VICReg_Lightcurves_2025-10-23_19-07-03',
        'cm_title': 'DiViT VICReg LC-MTA',
        'experiment_name': 'DiViT_VICReg LC',
        'max_time_to_eval': [8, 16, 32, 64, 128, 256, 512, 1024, 2048]
    },
    {   'type': 'lc',
        'name': 'DiViT_Periodic_VICReg_LinearClassification_2025-10-24_11-00-07',
        'cm_title': 'DiViT VICReg LC-MTA-LP',
        'experiment_name': 'DiViT_VICReg_Linear LC',
        'max_time_to_eval': [8, 16, 32, 64, 128, 256, 512, 1024, 2048]
    }
]

In [ ]:
experiments_mm = [
    {   'type': 'multimodal',
        'name': 'ATAT_Periodic_MM_2025-10-09_14-01-07',
        'cm_title': 'ATAT LC-MD-FT-MTA',
        'experiment_name': 'ATAT MM',
        'max_time_to_eval': [8, 16, 32, 64, 128, 256, 512, 1024, 2048]
    },
    {   'type': 'multimodal',
        'name': 'DiT_Periodic_MM_2025-10-09_18-22-05',
        'cm_title': 'DiT LC-MD-FT-MTA',
        'experiment_name': 'DiT MM',
        'max_time_to_eval': [8, 16, 32, 64, 128, 256, 512, 1024, 2048]
    },
    {   'type': 'multimodal',
        'name': 'DiT_Periodic_VICReg_MM_2025-10-13_07-41-01',
        'cm_title': 'DiT VICReg LC-MD-FT-MTA',
        'experiment_name': 'DiT_VICReg MM',
        'max_time_to_eval': [8, 16, 32, 64, 128, 256, 512, 1024, 2048]
    },
    {   'type': 'multimodal',
        'name': 'DiViT_L_Periodic_MM_2025-10-14_04-39-51',
        'cm_title': 'DiViT-L LC-MD-FT-MTA',
        'experiment_name': 'DiViT_L MM',
        'max_time_to_eval': [8, 16, 32, 64, 128, 256, 512, 1024, 2048]
    },
    {   'type': 'multimodal',
        'name': 'DiViT_L_Periodic_VICReg_MM_2025-10-14_07-18-37',
        'cm_title': 'DiViT-L VICReg LC-MD-FT-MTA',
        'experiment_name': 'DiViT_L_VICReg MM',
        'max_time_to_eval': [8, 16, 32, 64, 128, 256, 512, 1024, 2048]
    },
    {   'type': 'multimodal',
        'name': 'DiViT_Periodic_MM_2025-10-13_17-49-22',
        'cm_title': 'DiViT LC-MD-FT-MTA',
        'experiment_name': 'DiViT MM',
        'max_time_to_eval': [8, 16, 32, 64, 128, 256, 512, 1024, 2048]
    },
    {   'type': 'multimodal',
        'name': 'DiViT_Periodic_VICReg_MM_2025-10-13_19-46-30',
        'cm_title': 'DiViT VICReg LC-MD-FT-MTA',
        'experiment_name': 'DiViT_VICReg MM',
        'max_time_to_eval': [8, 16, 32, 64, 128, 256, 512, 1024, 2048]
    }
]

In [ ]:
experiments_lcmd = [
    {   'type': 'lc_md',
        'name': 'ATAT_Periodic_LC_MD_2025-10-18_04-11-11',
        'cm_title': 'ATAT LC-MD-MTA',
        'experiment_name': 'ATAT LC_MD',
        'max_time_to_eval': [8, 16, 32, 64, 128, 256, 512, 1024, 2048]
    },
    {   'type': 'lc_md',
        'name': 'DiT_Periodic_LC_MD_2025-10-16_08-22-17',
        'cm_title': 'DiT LC-MD-MTA',
        'experiment_name': 'DiT LC_MD',
        'max_time_to_eval': [8, 16, 32, 64, 128, 256, 512, 1024, 2048]
    },

    {   'type': 'lc_md',
        'name': 'DiT_Periodic_VICReg_LC_MD_2025-10-16_20-26-18',
        'cm_title': 'DiT VICReg LC-MD-MTA',
        'experiment_name': 'DiT_VICReg LC_MD',
        'max_time_to_eval': [8, 16, 32, 64, 128, 256, 512, 1024, 2048]
    },
    {   'type': 'lc_md',
        'name': 'DiViT_L_Periodic_LC_MD_2025-10-17_16-45-39',
        'cm_title': 'DiViT-L LC-MD-MTA',
        'experiment_name': 'DiViT_L LC_MD',
        'max_time_to_eval': [8, 16, 32, 64, 128, 256, 512, 1024, 2048]
    },
    {   'type': 'lc_md',
        'name': 'DiViT_L_Periodic_VICReg_LC_MD_2025-10-17_19-32-44',
        'cm_title': 'DiViT-L VICReg LC-MD-MTA',
        'experiment_name': 'DiViT_L_VICReg LC_MD',
        'max_time_to_eval': [8, 16, 32, 64, 128, 256, 512, 1024, 2048]
    },
    {   'type': 'lc_md',
        'name': 'DiViT_Periodic_LC_MD_2025-10-17_05-55-04',
        'cm_title': 'DiViT LC-MD-MTA',
        'experiment_name': 'DiViT LC_MD',
        'max_time_to_eval': [8, 16, 32, 64, 128, 256, 512, 1024, 2048]
    },
    {   'type': 'lc_md',
        'name': 'DiViT_Periodic_VICReg_LC_MD_2025-10-17_07-55-48',
        'cm_title': 'DiViT VICReg LC-MD-MTA',
        'experiment_name': 'DiViT_VICReg LC_MD',
        'max_time_to_eval': [8, 16, 32, 64, 128, 256, 512, 1024, 2048]
    }
]

In [ ]:
#for exp in experiments_lcmd:
#    run_several_exps( dict_exp=exp)

In [ ]:
for exp in experiments:
    run_several_exps( dict_exp=exp)

In [ ]:
for exp in experiments_mm:
    run_several_exps( dict_exp=exp)

: 